In [ ]:
# Import Data
import pandas as pd
import os
import plotly.express as px
import plotly.graph_objects as go

os.chdir(r'D:\Mission_Blitzkreig')

In [9]:
# Load your day 12 data
df = pd.read_csv('ooh_campaigns.csv')

# Recalculate the metrics from Day 12
df['ctr'] = (df['clicks'] / df['impressions'] * 100).round(2)
df['cpm'] = (df['spend'] / df['impressions'] / 1000).round(2)
df['roi'] = ((df['revenue'] - df['spend']) / df['spend'] * 100).round(2)
df['cvr'] = (df['conversions'] / df['clicks'] * 100).round(2)
df['profit'] = df['revenue'] - df['spend']

print("---Data loaded & Metrics Calculated---")

---Data loaded & Metrics Calculated---


In [26]:
# Group by city & revenue 
format_revenue = df.groupby('format')['revenue'].sum().reset_index()
format_revenue = format_revenue.sort_values('revenue', ascending=False)

In [28]:
# Create the pie chart 
fig =  px.pie(
    format_revenue,
    names='format',
    values='revenue',
    title='Total Revenue by Format',
    labels={'format':'Format','revenue':'Revenue'},
   color_discrete_sequence=px.colors.sequential.Greens,  # Changed this too
    hole=0.0  # Optional: set to 0.3 for a donut chart
)

# Add revenue values to the chart
fig.update_traces(textposition='inside', textinfo='percent+label')

fig.show()
fig.write_html('format_revenue.html')
print("✅ Chart saved: format_revenue.html")

✅ Chart saved: format_revenue.html


In [39]:
# 2. Create a scatter plot: CTR (x-axis) vs ROI (y-axis), colored by city

fig= px.scatter(
    df,
    x='ctr',
    y='roi',
    size='revenue',
    color='city',
    hover_data=['city','ctr','revenue'],
    labels={'ctr':'CTR' , 'roi': 'ROI'},
    color_discrete_sequence=px.colors.qualitative.Set2 
)
# Add diagonal line (break-even line)
fig.add_trace(go.Scatter(
    x=[0, df['ctr'].max()],
    y=[0, df['ctr'].max()],
    mode='lines',
    name='Break-even',
    line=dict(color='red', dash='dash')
))
 # Customize
fig.update_layout(
    xaxis_title="CTR",
    yaxis_title="ROI",
    font=dict(size=20),
    height=800
)

fig.show()
fig.write_html('ctr_vs_roi.html')
print("✅ Chart saved: ctr_vs_roi.html")


✅ Chart saved: ctr_vs_roi.html


In [41]:

# 3. Create a bar chart showing total conversions by city

# Group by city and sum conversions
city_conversions = df.groupby('city')['conversions'].sum().reset_index()
city_conversions = city_conversions.sort_values('conversions', ascending=False)

# Create bar chart
fig = px.bar(
    city_conversions,
    x='city',
    y='conversions',
    title='Total Conversions by City',
    labels={'city': 'City', 'conversions': 'Total Conversions'},
    color='conversions',
    color_continuous_scale='Blues',
    text='conversions'  # Show values on bars
)

# Format the text on bars
fig.update_traces(texttemplate='%{text:.0f}', textposition='outside')

# Customize layout
fig.update_layout(
    xaxis_title="City",
    yaxis_title="Total Conversions",
    font=dict(size=14),
    height=500
)

# Show and save
fig.show()
fig.write_html('conversions_by_city.html')
print("✅ Chart saved: conversions_by_city.html")

✅ Chart saved: conversions_by_city.html


In [42]:
# 4. Combine all charts into one HTML page with subplots

from plotly.subplots import make_subplots
import plotly.graph_objects as go

# Create a 2x2 grid of subplots
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'Total Revenue by Format',
        'CTR vs ROI by City',
        'Total Conversions by City',
        'Top 5 Campaigns by ROI'
    ),
    specs=[
        [{'type': 'pie'}, {'type': 'scatter'}],
        [{'type': 'bar'}, {'type': 'bar'}]
    ]
)

# ===== SUBPLOT 1: Pie Chart - Revenue by Format =====
format_revenue = df.groupby('format')['revenue'].sum().reset_index()

fig.add_trace(
    go.Pie(
        labels=format_revenue['format'],
        values=format_revenue['revenue'],
        name='Revenue by Format'
    ),
    row=1, col=1
)

# ===== SUBPLOT 2: Scatter - CTR vs ROI by City =====
for city in df['city'].unique():
    city_data = df[df['city'] == city]
    fig.add_trace(
        go.Scatter(
            x=city_data['ctr'],
            y=city_data['roi'],
            mode='markers',
            name=city,
            marker=dict(size=10)
        ),
        row=1, col=2
    )

# ===== SUBPLOT 3: Bar Chart - Conversions by City =====
conversions_by_city = df.groupby('city')['conversions'].sum().reset_index()
conversions_by_city = conversions_by_city.sort_values('conversions', ascending=False)

fig.add_trace(
    go.Bar(
        x=conversions_by_city['city'],
        y=conversions_by_city['conversions'],
        name='Conversions',
        marker_color='lightblue'
    ),
    row=2, col=1
)

# ===== SUBPLOT 4: Bar Chart - Top 5 Campaigns =====
top_5 = df.nlargest(5, 'roi')[['campaign_name', 'roi']]

fig.add_trace(
    go.Bar(
        x=top_5['roi'],
        y=top_5['campaign_name'],
        orientation='h',
        name='Top 5 ROI',
        marker_color='lightgreen'
    ),
    row=2, col=2
)

# ===== Update Layout =====
fig.update_layout(
    title_text="OOH Campaign Performance Dashboard",
    showlegend=True,
    height=900,
    width=1400,
    font=dict(size=12)
)

# Update axes labels
fig.update_xaxes(title_text="CTR (%)", row=1, col=2)
fig.update_yaxes(title_text="ROI (%)", row=1, col=2)
fig.update_xaxes(title_text="City", row=2, col=1)
fig.update_yaxes(title_text="Conversions", row=2, col=1)
fig.update_xaxes(title_text="ROI (%)", row=2, col=2)
fig.update_yaxes(title_text="Campaign", row=2, col=2)

# Show and save
fig.show()
fig.write_html('combined_dashboard.html')
print("✅ Dashboard saved: combined_dashboard.html")


✅ Dashboard saved: combined_dashboard.html
